In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/o4_mini_high_small/checkpoints/post_cell_11_try_6.pickle

In [ ]:
%%cudf.pandas.profile
### cell 12 ###

# Vectorized computation of gap_length without Python loops
prev_id = train['id'].shift(1)
prev_end = train['discourse_end'].shift(1)
curr_id = train['id']
curr_start = train['discourse_start']

cond_continuous = (curr_id == prev_id) & ((curr_start - prev_end) > 1)
cond_new_essay = (curr_id != prev_id) & (curr_start != 0)

train['gap_length'] = np.where(
    cond_continuous,
    curr_start - prev_end - 2,
    np.where(cond_new_essay, curr_start - 1, np.nan)
)

# Ensure the very first row matches the original hard-coded value
train.loc[train.index[0], 'gap_length'] = curr_start.iloc[0] - 1

# Compute gap at end of each essay and merge back
last_ones = train.drop_duplicates(subset='id', keep='last')
last_ones['gap_end_length'] = np.where(
    last_ones['discourse_end'] < last_ones['essay_len'],
    last_ones['essay_len'] - last_ones['discourse_end'],
    np.nan
)
cols_to_merge = ['id', 'discourse_id', 'gap_end_length']
train = train.merge(last_ones[cols_to_merge], on=['id', 'discourse_id'], how='left')